# Preprocess

In [53]:
%load_ext autoreload
%autoreload 2
 
import numpy as np
import os
from utils import load_emg_data, preprocess, sync_labels, sync_repetitions

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
os.chdir('/workspaces/TCC')
print("Working directory:", os.getcwd())

Working directory: /workspaces/TCC


## Parameters

In [55]:
WINDOW_MS, OVERLAP, LOWCUT = 200, 0.5, 20
FS_DB1, HIGHCUT_DB1 = 100,  45
FS_DB2, HIGHCUT_DB2 = 2000, 500

## Load

In [56]:
df1, stim1, restim1, rep1 = load_emg_data('Ninapro_DB1/s1.zip')
df2, stim2, restim2, rep2 = load_emg_data('Ninapro_DB2/DB2_s1.zip')

## Preprocess

In [57]:
Y1 = preprocess(df1, FS_DB1, LOWCUT, HIGHCUT_DB1, WINDOW_MS, OVERLAP)
Y2 = preprocess(df2, FS_DB2, LOWCUT, HIGHCUT_DB2, WINDOW_MS, OVERLAP)
 
print(f"Y1: {Y1.shape}  (windows, samples={Y1.shape[1]}, channels={Y1.shape[2]})")
print(f"Y2: {Y2.shape}  (windows, samples={Y2.shape[1]}, channels={Y2.shape[2]})")

Y1: (47147, 20, 10)  (windows, samples=20, channels=10)
Y2: (26192, 400, 12)  (windows, samples=400, channels=12)


### Labels from Restimulus (cleaner than stimulus)

In [58]:
y1 = sync_labels(restim1, FS_DB1, WINDOW_MS, OVERLAP)
y2 = sync_labels(restim2, FS_DB2, WINDOW_MS, OVERLAP)
r1 = sync_repetitions(rep1, FS_DB1, WINDOW_MS, OVERLAP)
r2 = sync_repetitions(rep2, FS_DB2, WINDOW_MS, OVERLAP)
 
assert Y1.shape[0] == y1.shape[0] == r1.shape[0], "DB1 shape mismatch"
assert Y2.shape[0] == y2.shape[0] == r2.shape[0], "DB2 shape mismatch"
 
print(f"\nDB1 active windows: {(y1>0).sum()} | transitions: {(y1==-1).sum()} | rest: {(y1==0).sum()}")
print(f"DB2 active windows: {(y2>0).sum()} | transitions: {(y2==-1).sum()} | rest: {(y2==0).sum()}")


DB1 active windows: 18108 | transitions: 0 | rest: 29039
DB2 active windows: 11982 | transitions: 0 | rest: 14210


## Save

In [59]:
os.makedirs('preprocessed_data', exist_ok=True)

for name, arr in [('EMG_Y1', Y1), ('EMG_Y2', Y2), ('LABELS_y1', y1), ('LABELS_y2', y2), ('REP_r1', r1), ('REP_r2', r2)]:
    np.save(f'preprocessed_data/{name}.npy', arr)

print("\nSaved: EMG_Y1, EMG_Y2, LABELS_y1, LABELS_y2, REP_r1, REP_r2 in preprocessed_data/")


Saved: EMG_Y1, EMG_Y2, LABELS_y1, LABELS_y2, REP_r1, REP_r2 in preprocessed_data/
